# How to build a basic Language Model
Today I want to talk about how a language model works and build one from scratch using PyTorch.
we will start from the very beginning, no magic, just simple code.

## Step 1: The Data
first we need some text to train on, let's use a few simple sentences

In [ ]:
text_1 = "I like Python programming."
text_2 = "Python is a great language for data science."
text_3 = "Many developers enjoy using Python for web development."
text_4 = "Python's simplicity and readability make it a popular choice among programmers."

# join all texts into one
text = text_1 + text_2 + text_3 + text_4
print(f"total characters: {len(text)}")

## Step 2: Tokenization
the computer can only understand numbers, so we need to transfer the plain text to numbers.
we call each character a **token**, and build a mapping between characters and integers.

In [ ]:
# get unique characters from all the texts
chars = sorted(set(text))  # sorted so the mapping is always the same
vocab_size = len(chars)
print(f"unique characters: {vocab_size}")
print(f"vocab: {''.join(chars)}")

In [ ]:
# create a mapping of characters to integers
stoi = {s: i for i, s in enumerate(chars)}
# create a mapping of integers to characters
itos = {i: s for s, i in stoi.items()}

# encode: text -> list of integers
encode = lambda text: [stoi[c] for c in text]
# decode: list of integers -> text
decode = lambda encoded: ''.join([itos[i] for i in encoded])

# test it
print(encode(text_1))
print(decode(encode(text_1)))  # should get back the original text

the code we defined above is called a **tokenizer** and each character is called a **token**

## Step 3: How does the model learn?
the first thing we need to know is how a model predicts the next character.
the idea is simple: given everything before, predict what comes next.

In [ ]:
# let's visualize what the training data looks like
for i in range(1, len(text_1)):
    x = text_1[:i]    # context: everything before
    y = text_1[i]     # target: the next character
    print(f"{x} -> {y}")

like the output above, we give the model all characters before and let it predict the next token.
now we need to build this dataset as PyTorch tensors.

## Step 4: Prepare the Dataset
we are using PyTorch to train our model, so we need to put our dataset into pytorch tensors.
the trick is simple: x is the text, y is the same text shifted by one character.

In [ ]:
import torch
# the nn module involves all neural network building blocks
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# encode the full text into a tensor
data = torch.tensor(encode(text_1), dtype=torch.long)

# x is the input (all but last character)
# y is the target (all but first character, shifted by one)
x = data[:-1]
y = data[1:]

print("input x:", x)
print("target y:", y)
print(f"\nfor example: token {x[0].item()} ({decode([x[0].item()])!r}) should predict token {y[0].item()} ({decode([y[0].item()])!r})")

## Step 5: Build the Bigram Model
the simplest possible model is a **Bigram** model.
it has just one layer: an embedding table.
given a token, it looks up a row in the table and that row is the prediction for the next token.

In [ ]:
class Bigram(nn.Module):
    # nn.Module gives us all the neural network building blocks
    def __init__(self, vocab_size):
        super().__init__()  # always call this, no arguments
        # the embedding table: for each token, store a row of vocab_size numbers
        # that row represents "what token is likely to come next"
        self.embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx):
        # idx is a tensor of token indices
        # embedding_table[idx] gives the prediction (logits) for each token
        logits = self.embedding_table(idx)  # (T, vocab_size)
        return logits

## Step 6: Train the Model
now we train the model by comparing its predictions to the actual next characters.
the loss tells us how wrong the model is, and we use it to update the weights.

In [ ]:
model = Bigram(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

for step in range(1000):
    logits = model(x)                          # get predictions
    loss = F.cross_entropy(logits, y)          # compare with targets
    optimizer.zero_grad(set_to_none=True)      # clear old gradients
    loss.backward()                            # compute new gradients
    optimizer.step()                           # update weights

    if step % 100 == 0:
        print(f"step {step} | loss: {loss.item():.4f}")

## Step 7: Generate Text
now let's see what the model learned! we give it a starting character and let it generate.

In [ ]:
def generate(model, start_char, max_new_tokens=50):
    # encode the starting character
    idx = torch.tensor(encode(start_char), dtype=torch.long)
    result = start_char

    for _ in range(max_new_tokens):
        logits = model(idx)           # get predictions
        logits = logits[-1, :]        # only care about the last token's prediction
        probs = F.softmax(logits, dim=-1)              # convert to probabilities
        next_token = torch.multinomial(probs, num_samples=1)  # sample next token
        result += decode([next_token.item()])          # add to result
        idx = torch.cat([idx, next_token])             # add to context

    return result

print(generate(model, 'I', max_new_tokens=100))

the output is not perfect, but the model learned something!
it knows which characters tend to follow which, just from the training text.

this is the simplest possible language model. in the next part we will add **attention** to make it much smarter.